# XAI Sensitivity & Stability — CPU vs GPU(L4) 비교

**런타임: L4 GPU** (런타임 → 런타임 유형 변경 → L4)

셀 순서대로 실행. 데이터zip / 가중치zip 은 **각각 따로** 업로드.

In [ ]:
# 1) 클론 + 패키지
%cd /content
!rm -rf fireimage_detection
!git clone https://github.com/yuntaewon812/fireimage_detection.git fireimage_detection -q
!pip install timm einops grad-cam lime shap scikit-image -q
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 2-A) 데이터 업로드 — fireimage_clean.zip (약 32MB) 하나만 선택
from google.colab import files
up1 = files.upload()
print('업로드:', list(up1.keys()))

In [ ]:
# 2-B) 가중치 업로드 — weights_E.zip (약 590MB) 하나만 선택
#  (용량이 커서 시간이 걸림. 끊기면 이 셀만 다시 실행)
from google.colab import files
up2 = files.upload()
print('업로드:', list(up2.keys()))

In [ ]:
# 3) 압축 해제
import zipfile, os
BASE = '/content/fireimage_detection'

if os.path.exists('/content/fireimage_clean.zip'):
    !python {BASE}/colab_setup.py
    print('데이터 압축 해제 완료')
else:
    print('주의: fireimage_clean.zip 없음 — 셀 2-A 다시 실행')

if os.path.exists('/content/weights_E.zip'):
    with zipfile.ZipFile('/content/weights_E.zip', 'r') as z:
        z.extractall(BASE)
    print('가중치 압축 해제 완료')
else:
    print('주의: weights_E.zip 없음 — 셀 2-B 다시 실행')

w_dir = f'{BASE}/model_save/fireimage_abl_E/fold0'
print('fold0 가중치:', os.listdir(w_dir) if os.path.exists(w_dir) else '없음')

In [ ]:
# 4) Sensitivity & Stability 실행 (GPU L4)
%cd /content/fireimage_detection
import time
t0 = time.time()
!python sensitivity_stability.py
gpu_total = time.time() - t0
print(f'\n총 GPU 실행 시간: {gpu_total:.1f}s ({gpu_total/60:.1f}분)')

In [ ]:
# 5) GPU(L4) 결과 표시 + (있으면) CPU 비교
import pandas as pd, os
gpu_df = pd.read_csv('results_SENS_STAB/sens_stab_cuda.csv')
print('=== GPU (L4) 결과 ===')
print(gpu_df[['Model','Method','Sensitivity','Stability','Time_sec']].to_string(index=False))

cpu_path = 'results_SENS_STAB/sens_stab_cpu.csv'
if os.path.exists(cpu_path):
    cpu_df = pd.read_csv(cpu_path)
    cmp = gpu_df[['Model','Method','Time_sec']].rename(columns={'Time_sec':'GPU_sec'})
    cmp = cmp.merge(cpu_df[['Model','Method','Time_sec']].rename(columns={'Time_sec':'CPU_sec'}),
                    on=['Model','Method'])
    cmp['Speedup'] = (cmp['CPU_sec'] / cmp['GPU_sec']).round(1)
    print('\n=== CPU vs GPU(L4) 속도 비교 ===')
    print(cmp.to_string(index=False))

In [ ]:
# (선택) 로컬 CPU 결과 업로드 — sens_stab_cpu.csv 올리면 셀5에서 비교됨
from google.colab import files
import shutil, os
up = files.upload()
for fn in up:
    if fn.endswith('.csv'):
        os.makedirs('results_SENS_STAB', exist_ok=True)
        shutil.move(fn, 'results_SENS_STAB/sens_stab_cpu.csv')
        print('업로드:', fn, '-> results_SENS_STAB/sens_stab_cpu.csv')

In [ ]:
# 6) 결과 다운로드
from google.colab import files
files.download('results_SENS_STAB/sens_stab_cuda.csv')